# LeetCode 1996 — The Number of Weak Characters in the Game bằng Python

Link đề: https://leetcode.com/problems/the-number-of-weak-characters-in-the-game/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Greedy, Sorting, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def numberOfWeakCharacters(self, properties: List[List[int]]) -> int:
        ...
```

Đếm số nhân vật **bị trội nghiêm ngặt** trên cả hai chỉ số (bài toán "dominance" 2 chiều). Notebook đi từ **so sánh mọi cặp `O(n²)`** → **sắp xếp tấn công giảm, phòng thủ tăng + max chạy** → **gom nhóm theo tấn công** → **đếm theo xô + max hậu tố `O(n + A)`** → **ngăn xếp đơn điệu**.

## 1. Mục tiêu học

1. Hiểu khái niệm **trội nghiêm ngặt** (`a_j > a_i` **và** `d_j > d_i`) và vì sao chữ "nghiêm ngặt" gây khó với các giá trị **bằng nhau**.
2. Dùng **sắp xếp với khóa hỗn hợp** `(-attack, defense)` để xử lý tấn công bằng nhau mà không cần gom nhóm.
3. Duy trì **max chạy** (prefix max) của phòng thủ để trả lời "có ai trội hơn" trong `O(1)`.
4. Dùng **max hậu tố theo giá trị** khi miền giá trị nhỏ (`≤ 10⁵`) để bỏ sắp xếp.
5. Nhận diện mô hình **ngăn xếp đơn điệu** cho cùng bài toán.

## 2. Đề bài & ví dụ

Mỗi nhân vật có `[attack, defense]`. Nhân vật `i` là **yếu** nếu tồn tại nhân vật `j` có `attack_j > attack_i` **và** `defense_j > defense_i`. Trả về số nhân vật yếu.

**Ví dụ 1:** `[[5,5],[6,3],[3,6]]` → `0` (không ai trội cả hai chỉ số so với ai).

**Ví dụ 2:** `[[2,2],[3,3]]` → `1` (`[2,2]` yếu vì `[3,3]`).

**Ví dụ 3:** `[[1,5],[10,4],[4,3]]` → `1` (`[4,3]` yếu vì `[10,4]`).

**Ràng buộc:** `2 ≤ n ≤ 10⁵`, `1 ≤ attack, defense ≤ 10⁵`.

In [1]:
import random
import time
from typing import List

examples = [([[5, 5], [6, 3], [3, 6]], 0), ([[2, 2], [3, 3]], 1), ([[1, 5], [10, 4], [4, 3]], 1)]

def dominators(props, i):
    a, d = props[i]
    return [j for j, (x, y) in enumerate(props) if x > a and y > d]

for props, exp in examples:
    print(props, "->", {tuple(props[i]): [tuple(props[j]) for j in dominators(props, i)] for i in range(len(props))}, "| đáp án", exp)

[[5, 5], [6, 3], [3, 6]] -> {(5, 5): [], (6, 3): [], (3, 6): []} | đáp án 0
[[2, 2], [3, 3]] -> {(2, 2): [(3, 3)], (3, 3): []} | đáp án 1
[[1, 5], [10, 4], [4, 3]] -> {(1, 5): [], (10, 4): [], (4, 3): [(10, 4)]} | đáp án 1


## 3. Kiến thức nền

### 3.1 Quan hệ trội (dominance) và tập Pareto

`j` trội `i` nếu `j` tốt hơn **nghiêm ngặt** ở cả hai chỉ số. Nhân vật **không yếu** là những người không bị ai trội nghiêm ngặt — gần giống "biên Pareto" (khác ở chỗ đây là trội **nghiêm ngặt** cả hai).

### 3.2 Sắp xếp với khóa nhiều thành phần

In [2]:
props = [[3, 6], [6, 3], [6, 5], [5, 5], [3, 2]]
print("sắp (-attack, defense):", sorted(props, key=lambda p: (-p[0], p[1])))
print("sắp (attack, -defense):", sorted(props, key=lambda p: (p[0], -p[1])))

sắp (-attack, defense): [[6, 3], [6, 5], [5, 5], [3, 2], [3, 6]]
sắp (attack, -defense): [[3, 6], [3, 2], [5, 5], [6, 5], [6, 3]]


### 3.3 Max chạy (prefix max)

Duyệt dãy, giữ `mx = max` các phần tử đã qua. "Có phần tử trước lớn hơn `x`?" ⇔ `mx > x`.

### 3.4 Max hậu tố theo giá trị

Nếu khóa là số nguyên nhỏ (`≤ A`), có thể lập `best[a]` = max phòng thủ của các nhân vật có tấn công **đúng bằng** `a`, rồi `suf[a] = max(best[a], suf[a+1])`. Truy vấn "max phòng thủ của người có tấn công `> a`" = `suf[a + 1]`.

In [3]:
best = [0] * 8
for a, d in [[3, 6], [6, 3], [6, 5], [5, 5], [3, 2]]:
    best[a] = max(best[a], d)
suf = [0] * 9
for a in range(7, -1, -1):
    suf[a] = max(best[a], suf[a + 1])
print("best:", best)
print("suf :", suf)

best: [0, 0, 0, 6, 0, 5, 5, 0]
suf : [6, 6, 6, 6, 5, 5, 5, 0, 0]


## 4. Ý tưởng nền tảng / trực giác

Nếu ta duyệt nhân vật theo **tấn công giảm dần**, thì khi đến nhân vật `i`, mọi người đã duyệt có tấn công **≥** `a_i`. Nếu chỉ những người có tấn công **>** `a_i` được tính, câu hỏi trở thành: "max phòng thủ của họ có lớn hơn `d_i` không?".

Rắc rối: người **cùng tấn công** cũng đã được duyệt. Mẹo: với cùng tấn công, sắp **phòng thủ tăng dần** — khi đó người cùng nhóm đứng trước có phòng thủ **≤** `d_i`, nên không làm `mx > d_i` sai lệch.

Ví dụ 3: sắp `(-a, d)`: `[10,4], [4,3], [1,5]`. `mx`: 4 → `[4,3]`: `3 < 4` yếu → `[1,5]`: `5 < 4`? không ⇒ 1.

## 5. Cách 1 — Brute Force / baseline: so sánh mọi cặp

In [4]:
def weak_brute(props):
    cnt = 0
    for a, d in props:
        if any(x > a and y > d for x, y in props):
            cnt += 1
    return cnt

for props_, exp in examples:
    print(weak_brute(props_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n²)` — `10¹⁰` với `n = 10⁵`: quá chậm. **Nút thắt:** với mỗi `i`, ta chỉ cần **một** con số: max phòng thủ trong số người có tấn công lớn hơn. Sắp xếp để tính nó một lần cho tất cả.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp theo `(-attack, defense)`** và duyệt, giữ `mx` là max phòng thủ đã gặp:
- **nếu** `d < mx` **thì** nhân vật này yếu: người có `mx` đứng trước nên tấn công `≥ a`; nếu tấn công **bằng** `a` thì phòng thủ của họ `≤ d` (vì cùng nhóm sắp tăng) — mâu thuẫn với `mx > d`; vậy tấn công **>** `a`.
- **ngược lại** cập nhật `mx = d`.

Không cần gom nhóm, không cần xử lý riêng trường hợp bằng nhau — tất cả nằm trong **khóa sắp xếp**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp `(-attack, defense)` + max chạy

In [5]:
def weak_sort(props):
    mx = 0; cnt = 0
    for a, d in sorted(props, key=lambda p: (-p[0], p[1])):
        if d < mx:
            cnt += 1
        else:
            mx = d
    return cnt

for props_, exp in examples:
    print(weak_sort(props_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ (bản sắp). Chỉ cập nhật `mx` ở nhánh `else` vẫn đúng vì nếu `d < mx` thì `max(mx, d) = mx`.

### 7.2 Cách 3 — Sắp tấn công giảm, gom nhóm theo tấn công

Cách tư duy "an toàn" khi chưa nghĩ ra mẹo khóa: xử lý **cả nhóm** cùng tấn công — so mọi thành viên với `mx` của các nhóm **trước**, rồi mới cập nhật `mx` bằng max của nhóm.

In [6]:
from itertools import groupby

def weak_groups(props):
    mx = 0; cnt = 0
    for a, grp in groupby(sorted(props, key=lambda p: -p[0]), key=lambda p: p[0]):
        ds = [d for _, d in grp]
        cnt += sum(d < mx for d in ds)          # chỉ so với nhóm tấn công lớn hơn
        mx = max(mx, max(ds))
    return cnt

for props_, exp in examples:
    print(weak_groups(props_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


### 7.3 Cách 4 — Đếm theo xô + max hậu tố `O(n + A)`

Miền tấn công `≤ 10⁵` ⇒ không cần sắp: `best[a]` = max phòng thủ với tấn công `a`, `suf[a]` = max của `best[a..A]`. Nhân vật `(a, d)` yếu ⇔ `d < suf[a + 1]`.

In [7]:
def weak_bucket(props):
    A = max(a for a, _ in props)
    best = [0] * (A + 2)
    for a, d in props:
        if d > best[a]:
            best[a] = d
    suf = [0] * (A + 2)
    for a in range(A, -1, -1):
        suf[a] = max(best[a], suf[a + 1])
    return sum(d < suf[a + 1] for a, d in props)

for props_, exp in examples:
    print(weak_bucket(props_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


**Độ phức tạp:** `O(n + A)` thời gian, `O(A)` bộ nhớ với `A = max attack ≤ 10⁵`.

### 7.4 Cách 5 — Ngăn xếp đơn điệu

Sắp `(attack, -defense)` (tấn công **tăng**, cùng tấn công thì phòng thủ **giảm**). Duyệt và giữ ngăn xếp phòng thủ **không tăng** từ đáy lên đỉnh: khi gặp `d`, mọi phần tử đỉnh `< d` bị **trội** bởi nhân vật hiện tại (tấn công lớn hơn nghiêm ngặt — vì cùng tấn công thì phòng thủ giảm, không thể `< d`) ⇒ pop và đếm.

In [8]:
def weak_stack(props):
    st = []; cnt = 0
    for a, d in sorted(props, key=lambda p: (p[0], -p[1])):
        while st and st[-1] < d:
            st.pop(); cnt += 1
        st.append(d)
    return cnt

for props_, exp in examples:
    print(weak_stack(props_), "(kỳ vọng", exp, ")")

0 (kỳ vọng 0 )
1 (kỳ vọng 1 )
1 (kỳ vọng 1 )


**Vì sao mỗi nhân vật yếu bị pop đúng một lần?** Nếu `i` bị trội bởi `j` (đứng sau `i` trong thứ tự tấn công tăng), thì khi đến `j` hoặc sớm hơn, `i` sẽ bị pop (các phần tử trên `i` trong ngăn xếp có phòng thủ `≤ d_i < d_j` nên cũng bị pop trước). Nếu `i` không bị ai trội thì không ai pop nó. Chi tiết ở mục 10.

## 8. TRACE từng bước

### 8.1 Trace cách 2 trên ví dụ tự dựng (đã đối chiếu vét cạn — đối chiếu lại đề gốc)

In [9]:
def trace_sort(props):
    mx = 0; cnt = 0
    order = sorted(props, key=lambda p: (-p[0], p[1]))
    print("  thứ tự sắp (-attack, defense):", order)
    for a, d in order:
        if d < mx:
            cnt += 1
            print(f"  [{a},{d}]: {d} < mx={mx} -> YẾU (tổng {cnt})")
        else:
            print(f"  [{a},{d}]: {d} ≥ mx={mx} -> không yếu, mx = {d}")
            mx = d
    return cnt

demo = [[7, 7], [7, 9], [5, 8], [5, 3], [3, 10], [2, 2], [7, 1]]
print("=>", trace_sort(demo), "| vét cạn:", weak_brute(demo))

  thứ tự sắp (-attack, defense): [[7, 1], [7, 7], [7, 9], [5, 3], [5, 8], [3, 10], [2, 2]]
  [7,1]: 1 ≥ mx=0 -> không yếu, mx = 1
  [7,7]: 7 ≥ mx=1 -> không yếu, mx = 7
  [7,9]: 9 ≥ mx=7 -> không yếu, mx = 9
  [5,3]: 3 < mx=9 -> YẾU (tổng 1)
  [5,8]: 8 < mx=9 -> YẾU (tổng 2)
  [3,10]: 10 ≥ mx=9 -> không yếu, mx = 10
  [2,2]: 2 < mx=10 -> YẾU (tổng 3)
=> 3 | vét cạn: 3


### 8.2 Trace cách 5 (ngăn xếp) trên cùng dữ liệu

In [10]:
def trace_stack(props):
    st = []; cnt = 0
    order = sorted(props, key=lambda p: (p[0], -p[1]))
    print("  thứ tự sắp (attack, -defense):", order)
    for a, d in order:
        popped = []
        while st and st[-1] < d:
            popped.append(st.pop()); cnt += 1
        st.append(d)
        note = f"pop {popped} (bị [{a},{d}] trội)" if popped else "không pop"
        print(f"  [{a},{d}]: {note:<28} stack = {st} | tổng {cnt}")
    return cnt

print("=>", trace_stack(demo))

  thứ tự sắp (attack, -defense): [[2, 2], [3, 10], [5, 8], [5, 3], [7, 9], [7, 7], [7, 1]]
  [2,2]: không pop                    stack = [2] | tổng 0
  [3,10]: pop [2] (bị [3,10] trội)     stack = [10] | tổng 1
  [5,8]: không pop                    stack = [10, 8] | tổng 1
  [5,3]: không pop                    stack = [10, 8, 3] | tổng 1
  [7,9]: pop [3, 8] (bị [7,9] trội)   stack = [10, 9] | tổng 3
  [7,7]: không pop                    stack = [10, 9, 7] | tổng 3
  [7,1]: không pop                    stack = [10, 9, 7, 1] | tổng 3
=> 3


### 8.3 Trace cách 4 (xô + max hậu tố)

In [11]:
def trace_bucket(props):
    A = max(a for a, _ in props)
    best = [0] * (A + 2)
    for a, d in props:
        best[a] = max(best[a], d)
    suf = [0] * (A + 2)
    for a in range(A, -1, -1):
        suf[a] = max(best[a], suf[a + 1])
    print("  a     :", list(range(A + 2)))
    print("  best  :", best)
    print("  suf   :", suf)
    for a, d in props:
        print(f"  [{a},{d}]: suf[{a + 1}] = {suf[a + 1]} -> {'YẾU' if d < suf[a + 1] else 'không yếu'}")
    return sum(d < suf[a + 1] for a, d in props)

print("=>", trace_bucket(demo))

  a     : [0, 1, 2, 3, 4, 5, 6, 7, 8]
  best  : [0, 0, 2, 10, 0, 8, 0, 9, 0]
  suf   : [10, 10, 10, 10, 9, 9, 9, 9, 0]
  [7,7]: suf[8] = 0 -> không yếu
  [7,9]: suf[8] = 0 -> không yếu
  [5,8]: suf[6] = 9 -> YẾU
  [5,3]: suf[6] = 9 -> YẾU
  [3,10]: suf[4] = 9 -> không yếu
  [2,2]: suf[3] = 10 -> YẾU
  [7,1]: suf[8] = 0 -> không yếu
=> 3


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def numberOfWeakCharacters(self, properties: List[List[int]]) -> int:
        properties.sort(key=lambda p: (-p[0], p[1]))      # tấn công giảm, cùng tấn công thì phòng thủ tăng
        mx = 0
        cnt = 0
        for _, d in properties:
            if d < mx:
                cnt += 1
            else:
                mx = d
        return cnt

for props_, exp in examples:
    assert Solution().numberOfWeakCharacters([p[:] for p in props_]) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


Ghi chú: code nộp **sắp tại chỗ** `properties` (LeetCode cho phép, tiết kiệm bộ nhớ). Nếu cần giữ nguyên đầu vào, dùng `sorted(...)` như cách 2.

## 10. Vì sao thuật toán đúng

**Cách 2.** Thứ tự duyệt: tấn công giảm; cùng tấn công, phòng thủ tăng. Tại bước của nhân vật `(a, d)`, `mx` là max phòng thủ của mọi người **đứng trước** (bất biến, vì nhánh `else` cập nhật `mx = d ≥ mx`).

- (⇒) Nếu `d < mx`: gọi `(a', d')` là người đạt `mx`, đứng trước nên `a' ≥ a`. Nếu `a' = a` thì vì cùng nhóm phòng thủ tăng, `d' ≤ d` — mâu thuẫn `d' = mx > d`. Vậy `a' > a`, `d' > d` ⇒ `(a, d)` yếu.
- (⇐) Nếu `(a, d)` yếu bởi `(a', d')` với `a' > a`, `d' > d` thì `(a', d')` đứng trước (tấn công lớn hơn) ⇒ `mx ≥ d' > d`.

**Cách 4.** `suf[a + 1]` = max phòng thủ của mọi người có tấn công `≥ a + 1`, tức `> a`. `(a, d)` yếu ⇔ tồn tại người tấn công `> a` với phòng thủ `> d` ⇔ `suf[a + 1] > d` (mặc định 0 < mọi phòng thủ ≥ 1).

**Cách 5.** Thứ tự: tấn công tăng, cùng tấn công phòng thủ giảm. Ngăn xếp luôn không tăng từ đáy lên. Khi nhân vật `j` pop `i` (`d_i < d_j`): `i` đứng trước `j`; nếu cùng tấn công thì `d_i ≥ d_j` (phòng thủ giảm) — mâu thuẫn; vậy `a_i < a_j` ⇒ `i` yếu. Ngược lại nếu `i` yếu bởi `j` thì `j` đứng sau `i`; khi `j` đến, nếu `i` còn trong ngăn xếp thì mọi phần tử trên `i` có phòng thủ `≤ d_i < d_j` ⇒ `j` pop tới `i`. Mỗi phần tử bị pop tối đa một lần ⇒ đếm chính xác.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | `O(n²)` | `O(1)` |
| 2. Sắp + max chạy | `O(n log n)` | `O(n)` (hoặc `O(1)` nếu sắp tại chỗ) |
| 3. Gom nhóm | `O(n log n)` | `O(n)` |
| 4. Xô + max hậu tố | `O(n + A)` | `O(A)` |
| 5. Ngăn xếp đơn điệu | `O(n log n)` | `O(n)` |

In [13]:
rnd = random.Random(1996)
big = [[rnd.randint(1, 10**5), rnd.randint(1, 10**5)] for _ in range(10**5)]
for name, fn in [("sort", weak_sort), ("groups", weak_groups), ("bucket", weak_bucket), ("stack", weak_stack),
                 ("Solution", lambda p: Solution().numberOfWeakCharacters([q[:] for q in p]))]:
    t0 = time.perf_counter(); r = fn(big); dt = time.perf_counter() - t0
    print(f"{name:<9}: {r}  {dt * 1000:7.1f} ms")

sort     : 99991     81.9 ms


groups   : 99991     63.9 ms
bucket   : 99991     26.1 ms
stack    : 99991     69.7 ms


Solution : 99991    114.0 ms


## 12. Bộ test

In [14]:
ALL = {
    "sort": weak_sort,
    "groups": weak_groups,
    "bucket": weak_bucket,
    "stack": weak_stack,
    "Solution": lambda p: Solution().numberOfWeakCharacters(p),
}

def check(props, exp=None, brute=True):
    ref = weak_brute(props) if brute else weak_sort(props)
    for name, fn in ALL.items():
        got = fn([p[:] for p in props])
        assert got == ref, (name, props, got, ref)
    if exp is not None:
        assert ref == exp, (props, ref, exp)
    return ref

for props_, exp in examples:
    check(props_, exp)
print("Ví dụ đề: OK")

check([[1, 1], [1, 2]], 0)                     # cùng tấn công: không ai yếu
check([[2, 2], [2, 2]], 0)                     # trùng hoàn toàn
check([[3, 3], [2, 3]], 0)                     # cùng phòng thủ
check([[1, 1], [2, 2], [3, 3], [4, 4]], 3)     # dây chuyền
check([[1, 4], [2, 3], [3, 2], [4, 1]], 0)     # phản xích
check([[10, 1], [5, 5], [1, 1]], 1)            # người trội không phải người tấn công cao nhất
check([[100000, 100000]] + [[99999, 99999]] * 5, 5)
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(3000):
    n = rnd.randint(2, 12); V = rnd.randint(1, 6)
    check([[rnd.randint(1, V), rnd.randint(1, V)] for _ in range(n)])
print("3000 test ngẫu nhiên nhỏ (nhiều giá trị trùng) so với vét cạn: OK")

for _ in range(8):
    n = rnd.randint(1000, 10**5); V = rnd.choice([10, 1000, 10**5])
    check([[rnd.randint(1, V), rnd.randint(1, V)] for _ in range(n)], brute=False)
print("8 test lớn (đối chiếu chéo): OK")

Ví dụ đề: OK
Edge case: OK
3000 test ngẫu nhiên nhỏ (nhiều giá trị trùng) so với vét cạn: OK


8 test lớn (đối chiếu chéo): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp cùng tấn công theo phòng thủ **giảm** (hoặc để mặc định giảm khi `reverse=True`)

In [15]:
def weak_bug_reverse(props):
    mx = 0; cnt = 0
    for a, d in sorted(props, reverse=True):        # SAI: (attack giảm, defense GIẢM)
        if d < mx: cnt += 1
        else: mx = d
    return cnt

props_ = [[1, 1], [1, 2]]
print("sai :", weak_bug_reverse(props_), "| đúng:", Solution().numberOfWeakCharacters([p[:] for p in props_]))

sai : 1 | đúng: 0


**Vì sao sai:** `[1,2]` đi trước làm `mx = 2`, rồi `[1,1]` bị coi là yếu — nhưng tấn công **bằng nhau** thì không trội nghiêm ngặt. **Sửa:** khóa `(-attack, defense)`.

### Lỗi 2 — Dùng `<=` thay vì `<`

In [16]:
def weak_bug_nonstrict(props):
    mx = 0; cnt = 0
    for a, d in sorted(props, key=lambda p: (-p[0], p[1])):
        if d <= mx: cnt += 1                         # SAI: phòng thủ bằng thì không yếu
        else: mx = d
    return cnt

props_ = [[3, 3], [2, 3]]
print("sai :", weak_bug_nonstrict(props_), "| đúng:", Solution().numberOfWeakCharacters([p[:] for p in props_]))

sai : 1 | đúng: 0


**Sửa:** điều kiện **nghiêm ngặt** `d < mx`.

### Lỗi 3 — Chỉ so với nhân vật có tấn công cao nhất

In [17]:
def weak_bug_max_attack_only(props):
    ta, td = max(props)                              # SAI: người trội không nhất thiết có tấn công cao nhất
    return sum(1 for a, d in props if a < ta and d < td)

props_ = [[10, 1], [5, 5], [1, 1]]
print("sai :", weak_bug_max_attack_only(props_), "| đúng:", Solution().numberOfWeakCharacters([p[:] for p in props_]))

sai : 0 | đúng: 1


**Vì sao sai:** `[1,1]` bị `[5,5]` trội, nhưng `[10,1]` (tấn công cao nhất) có phòng thủ thấp. **Sửa:** cần max phòng thủ trên **mọi** người tấn công lớn hơn — max chạy hoặc max hậu tố.

### Lỗi 4 — Max hậu tố lệch một: dùng `suf[a]` thay vì `suf[a + 1]`

In [18]:
def weak_bug_suffix(props):
    A = max(a for a, _ in props)
    best = [0] * (A + 2)
    for a, d in props: best[a] = max(best[a], d)
    suf = [0] * (A + 2)
    for a in range(A, -1, -1): suf[a] = max(best[a], suf[a + 1])
    return sum(d < suf[a] for a, d in props)        # SAI: suf[a] gồm cả người cùng tấn công

props_ = [[1, 1], [1, 2]]
print("sai :", weak_bug_suffix(props_), "| đúng:", Solution().numberOfWeakCharacters([p[:] for p in props_]))

sai : 1 | đúng: 0


**Sửa:** `suf[a + 1]` — chỉ những người tấn công **lớn hơn nghiêm ngặt**.

### Lỗi 5 — Ngăn xếp nhưng sắp cùng tấn công theo phòng thủ **tăng**

In [19]:
def weak_bug_stack_order(props):
    st = []; cnt = 0
    for a, d in sorted(props):                       # SAI: cùng tấn công phải phòng thủ GIẢM
        while st and st[-1] < d:
            st.pop(); cnt += 1
        st.append(d)
    return cnt

props_ = [[1, 1], [1, 2]]
print("sai :", weak_bug_stack_order(props_), "| đúng:", Solution().numberOfWeakCharacters([p[:] for p in props_]))

sai : 1 | đúng: 0


**Bài học chung:** mọi cách đều phải đảm bảo **người cùng tấn công không bao giờ "trội" nhau**. Mỗi thứ tự duyệt có một quy tắc phá hòa riêng — hãy tự hỏi "người cùng nhóm đứng trước có thể làm sai điều kiện không?".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tồn tại `j` lớn hơn nghiêm ngặt ở cả hai chiều" ⇒ sắp theo chiều 1, max chạy theo chiều 2.
- Phá hòa: **ngược chiều** với chiều sắp chính (`(-a, d)` hoặc `(a, -d)`).
- Miền giá trị nhỏ ⇒ đếm theo xô + max hậu tố, bỏ được `log n`.
- Đếm **số người trội** (không chỉ tồn tại) ⇒ sắp + **cây Fenwick** trên chiều 2 (bài tập 5).
- Tương tự: LeetCode 354 (Russian Doll Envelopes) dùng cùng mẹo phá hòa `(w, -h)`.

## 15. Biến thể của bài

1. **LeetCode 354 — Russian Doll Envelopes:** chuỗi lồng nhau dài nhất (LIS với phá hòa `(w, −h)`).
2. **LeetCode 406 — Queue Reconstruction by Height:** sắp `(−h, k)` rồi chèn.
3. **Trội không nghiêm ngặt** (`≥` cả hai, khác nhau ít nhất một): biên Pareto.
4. **Ba chỉ số:** dominance 3 chiều — sắp chiều 1, cây Fenwick/segment tree trên chiều 2 lưu max chiều 3.
5. **Đếm số người trội mỗi nhân vật:** sắp + Fenwick (bài tập 5).

In [20]:
def weak_3d_brute(props):
    return sum(any(x > a and y > b and z > c for x, y, z in props) for a, b, c in props)

class FenwickMax:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def update(self, i, v):                  # vị trí i (1-based) lấy max với v
        while i <= self.n:
            if v > self.t[i]: self.t[i] = v
            i += i & -i
    def query(self, i):                      # max trên [1, i]
        r = 0
        while i > 0:
            if self.t[i] > r: r = self.t[i]
            i -= i & -i
        return r

def weak_3d(props):
    """Biến thể 4: sắp chiều 1 giảm theo nhóm; Fenwick theo chiều 2 (đảo chiều) lưu max chiều 3."""
    B = max(b for _, b, _ in props)
    fw = FenwickMax(B)
    cnt = 0
    for a, grp in groupby(sorted(props, key=lambda p: -p[0]), key=lambda p: p[0]):
        grp = list(grp)
        for _, b, c in grp:                  # hỏi người có chiều 2 > b: chỉ số đảo B - b' + 1 với b' > b
            if fw.query(B - b) > c:
                cnt += 1
        for _, b, c in grp:
            fw.update(B - b + 1, c)
    return cnt

rnd_v = random.Random(15)
for _ in range(500):
    P3 = [[rnd_v.randint(1, 5), rnd_v.randint(1, 5), rnd_v.randint(1, 5)] for _ in range(rnd_v.randint(2, 10))]
    assert weak_3d(P3) == weak_3d_brute(P3)
print("Biến thể 3 chiều: OK, ví dụ", weak_3d([[1, 1, 1], [2, 2, 2], [3, 1, 3], [2, 3, 1]]))

Biến thể 3 chiều: OK, ví dụ 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | định nghĩa | `O(n²)` | `O(1)` | kiểm thử |
| 2. Sắp `(-a, d)` + max | phá hòa trong khóa | `O(n log n)` | `O(1)`–`O(n)` | **nộp bài** |
| 3. Gom nhóm | so với nhóm trước | `O(n log n)` | `O(n)` | dễ chứng minh |
| 4. Xô + max hậu tố | miền giá trị nhỏ | `O(n + A)` | `O(A)` | nhanh nhất về lý thuyết |
| 5. Ngăn xếp đơn điệu | pop người bị trội | `O(n log n)` | `O(n)` | luyện monotonic stack |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().numberOfWeakCharacters([[1,1],[2,1],[2,2],[1,2]])`?

**Bài 2 (tính tay):** Sắp `[[4,4],[4,7],[6,2],[2,9],[1,1]]` theo `(-attack, defense)` và chạy cách 2 bằng tay.

**Bài 3 (điền TODO):** Hoàn thiện `strong_characters(props)` — trả về **danh sách** (đã sắp) các nhân vật **không yếu**.

**Bài 4 (tự viết lại):** Viết lại cách 3 (gom nhóm) **không dùng `groupby`**, bằng hai con trỏ trên mảng đã sắp.

**Bài 5 (mini challenge):** Với mỗi nhân vật, đếm **số người trội nghiêm ngặt** nó; trả về danh sách theo thứ tự đầu vào. Yêu cầu `O(n log n)` (sắp + Fenwick đếm).

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def strong_characters(props):
    # TODO: như cách 2, nhưng thu thập các nhân vật không yếu
    pass

# Bài 4 — TODO
def weak_groups_two_pointers(props):
    # TODO: i chạy đầu nhóm, j tìm cuối nhóm cùng attack
    pass

# Bài 5 — TODO
def count_dominators(props):
    # TODO: sắp attack giảm theo nhóm; Fenwick đếm số defense > d đã thêm
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Ai có tấn công **và** phòng thủ lớn hơn `[1,1]`?
- Mức 2: Chỉ `[2,2]`.
- Mức 3: Đáp án `1`.

**Bài 2**
- Mức 1: Thứ tự `[6,2], [4,4], [4,7], [2,9], [1,1]`.
- Mức 2: `mx`: 2 → 4 → 7 → 9; `[1,1]`: `1 < 9` yếu.
- Mức 3: Đáp án `1`.

**Bài 3**
- Mức 1: Nhánh `else` là nhân vật không yếu.
- Mức 2: Thu thập `[a, d]` vào danh sách.
- Mức 3: Trả `sorted(...)` để so sánh dễ.

**Bài 4**
- Mức 1: Sắp theo `-a`.
- Mức 2: `j = i; while j < n and a[j] == a[i]: j += 1`.
- Mức 3: Đếm trong `[i, j)` với `mx` cũ, rồi cập nhật `mx`.

**Bài 5**
- Mức 1: Duyệt nhóm tấn công giảm dần.
- Mức 2: Fenwick đếm số phòng thủ đã thêm; số `> d` = tổng − `prefix(d)`.
- Mức 3: Hỏi cả nhóm trước, thêm cả nhóm sau.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
r1 = Solution().numberOfWeakCharacters([[1, 1], [2, 1], [2, 2], [1, 2]])
print("Bài 1:", r1); assert r1 == 1 == weak_brute([[1, 1], [2, 1], [2, 2], [1, 2]])

# Bài 2
b2 = [[4, 4], [4, 7], [6, 2], [2, 9], [1, 1]]
print("Bài 2:"); r2 = trace_sort(b2); assert r2 == 1 == weak_brute(b2)

# Bài 3
def strong_characters_sol(props):
    mx = 0; out = []
    for a, d in sorted(props, key=lambda p: (-p[0], p[1])):
        if d >= mx:
            out.append([a, d]); mx = d
    return sorted(out)

rnd = random.Random(3)
for _ in range(1000):
    P = [[rnd.randint(1, 5), rnd.randint(1, 5)] for _ in range(rnd.randint(2, 10))]
    exp_strong = sorted(p for p in P if not any(x > p[0] and y > p[1] for x, y in P))
    assert strong_characters_sol(P) == exp_strong
print("Bài 3 OK, ví dụ:", strong_characters_sol(b2))

Bài 1: 1
Bài 2:
  thứ tự sắp (-attack, defense): [[6, 2], [4, 4], [4, 7], [2, 9], [1, 1]]
  [6,2]: 2 ≥ mx=0 -> không yếu, mx = 2
  [4,4]: 4 ≥ mx=2 -> không yếu, mx = 4
  [4,7]: 7 ≥ mx=4 -> không yếu, mx = 7
  [2,9]: 9 ≥ mx=7 -> không yếu, mx = 9
  [1,1]: 1 < mx=9 -> YẾU (tổng 1)
Bài 3 OK, ví dụ: [[2, 9], [4, 4], [4, 7], [6, 2]]


In [23]:
# Bài 4
def weak_groups_two_pointers_sol(props):
    a = sorted(props, key=lambda p: -p[0]); n = len(a)
    i = 0; mx = 0; cnt = 0
    while i < n:
        j = i
        while j < n and a[j][0] == a[i][0]:
            j += 1
        cnt += sum(1 for k in range(i, j) if a[k][1] < mx)
        mx = max(mx, max(a[k][1] for k in range(i, j)))
        i = j
    return cnt

# Bài 5
class FenwickSum:
    def __init__(self, n):
        self.n = n; self.t = [0] * (n + 1)
    def add(self, i, v=1):
        while i <= self.n: self.t[i] += v; i += i & -i
    def prefix(self, i):
        s = 0
        while i > 0: s += self.t[i]; i -= i & -i
        return s

def count_dominators_sol(props):
    D = max(d for _, d in props)
    fw = FenwickSum(D); total = 0
    res = [0] * len(props)
    idx = sorted(range(len(props)), key=lambda i: -props[i][0])
    k = 0
    while k < len(idx):
        g = k
        while g < len(idx) and props[idx[g]][0] == props[idx[k]][0]:
            g += 1
        for t in range(k, g):
            d = props[idx[t]][1]
            res[idx[t]] = total - fw.prefix(d)       # số người đã thêm (tấn công lớn hơn) có phòng thủ > d
        for t in range(k, g):
            fw.add(props[idx[t]][1]); total += 1
        k = g
    return res

rnd = random.Random(5)
for _ in range(1500):
    P = [[rnd.randint(1, 6), rnd.randint(1, 6)] for _ in range(rnd.randint(2, 12))]
    assert weak_groups_two_pointers_sol(P) == weak_brute(P)
    assert count_dominators_sol(P) == [len(dominators(P, i)) for i in range(len(P))]
print("Bài 4, Bài 5 OK; ví dụ đếm người trội:", count_dominators_sol([[1, 1], [2, 2], [3, 3], [2, 1]]))

Bài 4, Bài 5 OK; ví dụ đếm người trội: [2, 1, 0, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template dominance 2 chiều (tồn tại người trội nghiêm ngặt):**

```python
mx = cnt = 0
for a, d in sorted(props, key=lambda p: (-p[0], p[1])):   # phá hòa NGƯỢC chiều
    if d < mx: cnt += 1
    else: mx = d
```

**Checklist:**
- [ ] Phá hòa cùng tấn công đúng chiều (`(-a, d)` hoặc `(a, -d)` cho ngăn xếp)?
- [ ] So sánh **nghiêm ngặt** `<`?
- [ ] Max hậu tố dùng `suf[a + 1]`?
- [ ] Test có nhiều giá trị trùng (cùng tấn công, cùng phòng thủ, trùng hoàn toàn)?
- [ ] Không so mỗi người chỉ với người tấn công cao nhất?
- [ ] Chữ ký đúng: `def numberOfWeakCharacters(self, properties: List[List[int]]) -> int`?